# PMCortex Retrieval Dataset Pipeline

This notebook builds a biomedical document-retrieval benchmark from PMC citation contexts. Papers cited in a query sentence are positive documents; other references from the same source article form topically close hard-negative candidates.

Run the sections in order. Data products are written below `data/pmc_context_queries_v0-5/` and are intentionally not tracked by Git.


## 1. Discover seed articles with MeSH

Extract coarse MeSH categories from the local descriptor file and use their names as PMC search queries. The resulting PMCID list defines the source-article collection.


In [ ]:
import pandas as pd
import tqdm

from pmcortex.mesh import extract_mesh_coarsest_descriptor_categories
from pmcortex.pmcsearch import PMCSearch

# define a set of MeSH categories to use as search queries, based on the coarsest descriptor categories in the MeSH hierarchy
cats = extract_mesh_coarsest_descriptor_categories("../data/desc2026.xml", level=2)
use_keywords = {
    c["tree_number"]: c["name"]
    for c in cats
    if c["tree_number"][0] in ['A', 'B', 'C', 'D', 'E', 'G']
}

# use the MeSH categories as search queries to get seed PMC articles
seed_pmcs = []
with PMCSearch(
    user_agent="PMCortex/1.0 (contact: lorenz@hexemer.eu)",
    max_requests_per_second=2.9,
    max_attempts=3,
    retry_backoff_s=1.0,
) as search:
    for mesh, query in tqdm.tqdm(use_keywords.items()):
        df_ = search.search(query=f"({query}) AND open access[filter]", max_results=20)
        df_ = df_.assign(query=query, mesh=mesh)
        seed_pmcs.append(df_)

# combine and filter the seed PMC articles
seed_pmcs = pd.concat(seed_pmcs)
filtered_seeds = (
    seed_pmcs
    .query('not title.str.contains("correction", case=False)')
    .query('pmcid.str.startswith("PMC")')
    .groupby(['pmcid', 'title'])
    .agg(list)
    .reset_index()
)
filtered_seeds.to_csv("../data/mesh_based_pmc_seed_openaccess.csv", index=False)

## 2. Download PMC articles as JATS XML

Download the seed articles through PMC OAI-PMH. Existing files are reused, and unavailable articles are represented by their download status.


In [24]:
import pandas as pd
from pmcortex.downloader import PMCJATSDownloader

filtered_seeds = pd.read_csv("../data/mesh_based_pmc_seed_openaccess.csv")
required_seeds = filtered_seeds['pmcid'].unique()

In [ ]:
with PMCJATSDownloader(
    min_interval_s=0.6,
    out_dir="../data/pmc_context_queries_v1/xml_jats",
    user_agent="PMCortex/0.2 (contact: lorenz@hexemer.eu)",
    max_retries=3,
) as dl:
    results = dl.download_many(required_seeds[:100])

### Download audit

Inspect download volume and mark which seed articles have a local JATS document.


In [ ]:
import os
total_size = 0
counter = 0
for file in os.scandir('../data/pmc_context_queries_v0-5/xml_jats'):
    if file.is_file() and file.name.endswith('.nxml'):
        file_info = file.stat()
        total_size += file_info.st_size
        counter += 1
print(f"Total size: {total_size / (1024 * 1024):.2f} MB")
print(f"Number of files: {counter}")
print(f"Average size per file: {total_size / counter / 1024:.2f} KB")

## 3. Extract retrieval queries and references

Parse each JATS document into article metadata, bibliography entries, citation-derived queries, positioned full-text sentences, and normalized author data. Query rows and full-text lines share `source_pmcid`, `section_index`, `paragraph_index`, and `sentence_index` identifiers.


In [ ]:
from pathlib import Path
import yaml
import os
import tqdm

from pmcortex.jatsparser import JATSParser

abstracts = Path('../data/pmc_context_queries_v0-5/extracted_metadata.csv')
abstracts.unlink(missing_ok=True)
contexts = Path('../data/pmc_context_queries_v0-5/extracted_contexts.csv')
contexts.unlink(missing_ok=True)
sources = Path('../data/pmc_context_queries_v0-5/extracted_sources.csv')
sources.unlink(missing_ok=True)
text_path = Path('../data/pmc_context_queries_v0-5/fulltexts/')
text_path.mkdir(exist_ok=True, parents=True)
author_path = Path('../data/pmc_context_queries_v0-5/authors/')
author_path.mkdir(exist_ok=True, parents=True)



for file in tqdm.tqdm(os.scandir("../data/pmc_context_queries_v0-5/xml_jats")):
    
    if file.name.endswith(".nxml"):
        parser = JATSParser()
        article = parser.parse_article(file.path, save_sentences=True)
        
        sources_df = parser.sources_to_dataframe()
        sources_df = sources_df.assign(source_pmcid=article.pmcid)
        sources_df.to_csv(
            str(sources),
            mode='a' if sources.exists() else 'w',
            header=not sources.exists(),
            index=None
        )

        context_df = parser.contexts_to_dataframe()
        context_df.to_csv(
            str(contexts),
            mode='a' if contexts.exists() else 'w',
            header=not contexts.exists(),
            index=None
        )

        abstract_df = parser.metadata_to_dataframe()
        abstract_df.to_csv(
            str(abstracts),
            mode='a' if abstracts.exists() else 'w',
            header=not abstracts.exists(),
            index=None
        )

        fulltext = parser.text_to_list()
        with open(text_path / file.name.replace('.nxml', '.txt'), 'w') as fp:
            fp.writelines(fulltext)


        authors_and_refs = parser.normalized_authors_and_refs()
        with open(author_path / file.name.replace('.nxml', ".yaml"), 'w', encoding="utf-8") as fp:
            fp.write(
                yaml.safe_dump(authors_and_refs, sort_keys=False, allow_unicode=False),
            )   

### Extraction audit

Summarize generated full-text and author files to catch incomplete extraction runs.


In [ ]:
import os
total_size = 0
counter = 0
for file in os.scandir('../data/pmc_context_queries_v0-5/fulltexts'):
    if file.is_file() and file.name.endswith('.txt'):
        file_info = file.stat()
        total_size += file_info.st_size
        counter += 1
print(f"Total size: {total_size / (1024 * 1024):.2f} MB")
print(f"Number of files: {counter}")
print(f"Average size per file: {total_size / counter / 1024:.2f} KB")

In [ ]:
import os
total_size = 0
counter = 0
for file in os.scandir('../data/pmc_context_queries_v0-5/authors'):
    if file.is_file() and file.name.endswith('.yaml'):
        file_info = file.stat()
        total_size += file_info.st_size
        counter += 1
print(f"Total size: {total_size / (1024 * 1024):.2f} MB")
print(f"Number of files: {counter}")
print(f"Average size per file: {total_size / counter / 1024:.2f} KB")

## 4. Apply deterministic query filters

Remove clear formatting problems, explicit citation spoilers, self-citations, and queries with insufficient reference candidates. The self-contained checks are deliberately conservative; ambiguous cross-sentence anaphora is handled later by the Coreference model.


In [ ]:
from pmcortex.query_filters import QueryFilterPipeline

pipeline = QueryFilterPipeline("../data/pmc_context_queries_v0-5")
print(f"Remaining rows: {len(pipeline.filtered_df)}")

# Remove explicit document-context dependencies; cross-sentence anaphora is handled later.
pipeline.filter_self_contained_queries()
# pipeline.filter_no_summary_starts()
pipeline.filter_max_strangeness(0.3)
print(f"Remaining rows {len(pipeline.filtered_df)} after base filters")

# Reference-count filters define the benchmark candidate pool.
pipeline.filter_min_hit_queries(1)
pipeline.filter_min_source_count(20)
print(f"Remaining rows {len(pipeline.filtered_df)} after hit and source count filters")

# Remove explicit spoilers and self-citations.
pipeline.filter_no_et_al_queries()
pipeline.filter_no_explicit_spoilers()
pipeline.filter_no_self_citations()
print(f"Remaining rows {len(pipeline.filtered_df)} after spoiler and self-citation filters")

pipeline.cleanup_queries()
pipeline.filter_min_query_length(6)
pipeline.filter_max_query_length(100)
print(f"Remaining rows {len(pipeline.filtered_df)} after cleanup and length filters")

df_queries = pipeline.filtered_df[[
    'query',
    'context',
    'source_pmcid',
    'section_index',
    'paragraph_index',
    'sentence_index',
    'hits_parsed',
    'n_sources',
]].rename(columns={'hits_parsed': 'hits'})


## 5. Build the hard-negative benchmark

Select queries from source articles with one or two eligible queries. For every query, choose references cited elsewhere in the same article but not in the query sentence.


In [ ]:
relevant_pmcids = pipeline.filtered_df.value_counts('source_pmcid').reset_index().query('1 <= count <= 2')['source_pmcid'].tolist()
selected_df, selected_refs = pipeline.select_shared_reference_subset(n=19, relevant_pmcids=relevant_pmcids)
len(selected_df), len(selected_refs)

In [ ]:
selected_df[[
    'source_pmcid',
    'section_index',
    'paragraph_index',
    'sentence_index',
    'context',
    'query',
    'hits_parsed',
    'selected_decoys',
]].to_csv("../data/pmc_context_queries_v0-5/benchmark_queries_1-2.csv", index=False)

## 6. Collect Abstracts and MeSH data

Fetch PubMed abstracts and MeSH annotations for every positive and hard-negative PMID. Requests are processed in batches and appended to local CSV stores.


In [ ]:
import tqdm
import pandas as pd
from pathlib import Path
from pmcortex import PubMedMeSHClient, extract_abstracts_from_pubmed_xml, extract_mesh_terms_from_pubmed_xml

ABSTACT_DB = Path("../data/pmc_context_queries_v0-5/pmid_abstracts.csv")
MESH_DB = Path("../data/pmc_context_queries_v0-5/pmid_mesh_terms.csv")
ABSTACT_DB.unlink(missing_ok=True)
MESH_DB.unlink(missing_ok=True)

selected_df = pd.read_csv("../data/pmc_context_queries_v0-5/benchmark_queries_1-2.csv")
required_pmids = (
    set(selected_df['selected_decoys'].apply(eval).explode())
    .union(set(selected_df['hits_parsed'].apply(eval).explode()))
)


with PubMedMeSHClient() as client:
    for i in tqdm.tqdm(range(0, len(required_pmids), 1000), desc="Fetching PubMed metadata"):
        batch_pmids = list(required_pmids)[i:i+1000]
        xml = client.fetch_pubmed_xml(batch_pmids)
        abstracts = extract_abstracts_from_pubmed_xml(xml)
        mesh_terms = extract_mesh_terms_from_pubmed_xml(xml)
        
        abstracts.to_csv(ABSTACT_DB, mode='a', header=not ABSTACT_DB.exists(), index=False)
        mesh_terms.to_csv(MESH_DB, mode='a', header=not MESH_DB.exists(), index=False)

## 7. Remove cross-context coreferences

Use FastCoref on `context + query` and remove a query only when an anaphoric mention in the query resolves to a mention in the preceding context. This is more selective than rejecting every query that merely contains a pronoun.

The language model must be installed explicitly. This notebook never downloads a model as a hidden side effect.


In [ ]:
import spacy
from fastcoref import spacy_component  # Registers the FastCoref spaCy component.

from pmcortex import CrossContextCoreferenceFilter

SPACY_MODEL = "en_core_web_sm"
try:
    coref_nlp = spacy.load(SPACY_MODEL)
except OSError as error:
    raise RuntimeError(
        f"Install the spaCy model first: python -m spacy download {SPACY_MODEL}"
    ) from error

if "fastcoref" not in coref_nlp.pipe_names:
    coref_nlp.add_pipe("fastcoref")

cross_context_filter = CrossContextCoreferenceFilter(
    coref_nlp,
    batch_size=512,
)


In [ ]:
from pathlib import Path
import pandas as pd

BENCHMARK_DB = Path(
    "../data/pmc_context_queries_v0-5/benchmark_queries_1-2.csv"
)
COREF_FILTERED_DB = Path(
    "../data/pmc_context_queries_v0-5/benchmark_queries_1-2_coref_filtered.csv"
)

selected_df = pd.read_csv(BENCHMARK_DB)
rows_before_coreference = len(selected_df)
selected_df = cross_context_filter.filter_cross_context_coreferences(selected_df)
selected_df.to_csv(COREF_FILTERED_DB, index=False)

print(
    f"Removed {rows_before_coreference - len(selected_df)} cross-context queries; "
    f"{len(selected_df)} remain."
)


## 8. Compare hard and soft negatives with MeSH

Load the coreference-filtered benchmark, calculate MeSH overlap for same-article hard negatives, and sample cross-article soft negatives for comparison.


In [ ]:
import pandas as pd
from pathlib import Path

MESH_DB = Path("../data/pmc_context_queries_v0-5/pmid_mesh_terms.csv")
QUERY_DB = Path("../data/pmc_context_queries_v0-5/benchmark_queries_1-2_coref_filtered.csv")

all_mesh = pd.read_csv(MESH_DB)
all_mesh['pmid'] =  all_mesh['pmid'].astype(str)
selected_df = pd.read_csv(QUERY_DB)
selected_df['selected_decoys'] = selected_df['selected_decoys'].apply(eval)
selected_df['hits_parsed'] = selected_df['hits_parsed'].apply(lambda x: eval(x)[0])

In [ ]:
selected_df['soft_decoys'] = None
selected_df['selected_decoy_similarities'] = None
selected_df['soft_decoy_similarities'] = None
selected_df['n_hit_meshs'] = None

for i, row in tqdm.tqdm(selected_df.iterrows(), total=len(selected_df), desc="Calculating MeSH similarities"):
    pmcid = row['source_pmcid']
    hit_pmid = row['hits_parsed']
    other_pmcids = selected_df.query('source_pmcid != @pmcid')
    other_pmids = other_pmcids.sample(5)['selected_decoys'].explode().sample(80)

    hit_meshs = all_mesh.query('pmid == @hit_pmid')['descriptor_name'].to_list()
    if len(hit_meshs) == 0:
        continue

    decoy_meshs = [all_mesh.query('pmid == @decoy')['descriptor_name'].to_list() for decoy in row['selected_decoys']]
    soft_decoy_meshs = [all_mesh.query('pmid == @decoy')['descriptor_name'].to_list() for decoy in other_pmids]
    
    decoy_mesh_similarities = [len(set(d_mesh).intersection(set(hit_meshs)))/len(hit_meshs) for d_mesh in decoy_meshs]
    soft_decoy_mesh_similarities = [len(set(d_mesh).intersection(set(hit_meshs)))/len(hit_meshs)for d_mesh in soft_decoy_meshs]
    row['selected_decoy_similarities'] = decoy_mesh_similarities
    row['soft_decoy_similarities'] = soft_decoy_mesh_similarities
    row['soft_decoys'] = other_pmids.tolist()
    row['n_hit_meshs'] = len(hit_meshs)
    selected_df.loc[i] = row

In [ ]:
selected_df.to_csv("../data/pmc_context_queries_v0-5/benchmark_queries_1-2_softdecoys.csv", index=False)

In [ ]:
selected_df

## 9. Explore alternative candidate-pool sizes

Inspect how many queries remain when source articles contribute larger query groups. These cells are exploratory and do not overwrite the main benchmark.


In [ ]:
relevant_pmcids = pipeline.filtered_df.value_counts('source_pmcid').reset_index().query('2 <= count <= 3')['source_pmcid'].tolist()
selected_df, selected_refs = pipeline.select_shared_reference_subset(n=19, relevant_pmcids=relevant_pmcids)
len(selected_df), len(selected_refs)

In [ ]:
relevant_pmcids = pipeline.filtered_df.value_counts('source_pmcid').reset_index().query('3 <= count <= 5')['source_pmcid'].tolist()
selected_df, selected_refs = pipeline.select_shared_reference_subset(n=19, relevant_pmcids=relevant_pmcids)
len(selected_df), len(selected_refs)

In [ ]:
relevant_pmcids = pipeline.filtered_df.value_counts('source_pmcid').reset_index().query('7 < count < 10')['source_pmcid'].tolist()
len(pipeline.filtered_df.query('source_pmcid in @relevant_pmcids'))

In [ ]:
relevant_pmcids = pipeline.filtered_df.value_counts('source_pmcid').reset_index().query('8 < count < 12')['source_pmcid'].tolist()
len(pipeline.filtered_df.query('source_pmcid in @relevant_pmcids'))